# Description

(Please, take a look at the README.md file in this directory for instructions on how to run this notebook)

This notebook computes the covariance for each chromosome of all variants present in prediction models.

It has specicfic parameters for papermill (see under `Settings` below).

This notebook is not directly run. See README.md.

# Modules

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import conf
from entity import Gene

# Settings

In [3]:
# reference panel such as 1000G or GTEX_V8
REFERENCE_PANEL = None

# predictions models such as MASHR or ELASTIC_NET
EQTL_MODEL = None

# the numpy dtype used for the covariance matrix
#  either float64 or float32 (for huge matrices)
COVARIANCE_MATRIX_DTYPE = None

In [4]:
# Parameters
PHENOPLIER_NOTEBOOK_FILEPATH = "nbs/15_gsa_gls/05-snps_into_chr_cov.ipynb"
REFERENCE_PANEL = "GTEX_V8"
EQTL_MODEL = "MASHR"


In [5]:
assert (
    REFERENCE_PANEL is not None and len(REFERENCE_PANEL) > 0
), "A reference panel must be given"
display(f"Reference panel: {REFERENCE_PANEL}")

REFERENCE_PANEL_DIR = conf.PHENOMEXCAN["LD_BLOCKS"][f"{REFERENCE_PANEL}_GENOTYPE_DIR"]
display(f"Using reference panel folder: {str(REFERENCE_PANEL_DIR)}")
assert REFERENCE_PANEL_DIR.exists(), "Reference panel folder does not exist"

'Reference panel: GTEX_V8'

'Using reference panel folder: /home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/data/twas/predixcan/reference_panels/gtex_v8'

In [6]:
assert (
    EQTL_MODEL is not None and len(EQTL_MODEL) > 0
), "A prediction/eQTL model must be given"

EQTL_MODEL_FILES_PREFIX = conf.PHENOMEXCAN["PREDICTION_MODELS"][f"{EQTL_MODEL}_PREFIX"]

display(f"Using eQTL model: {EQTL_MODEL} / {EQTL_MODEL_FILES_PREFIX}")

'Using eQTL model: MASHR / mashr_'

In [7]:
OUTPUT_DIR_BASE = (
    conf.RESULTS["GLS"]
    / "gene_corrs"
    / "reference_panels"
    / REFERENCE_PANEL.lower()
    / EQTL_MODEL.lower()
)
OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)

In [8]:
display(f"Using output dir base: {OUTPUT_DIR_BASE}")

'Using output dir base: /home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/gene_corrs/reference_panels/gtex_v8/mashr'

In [9]:
cov_dtype_dict = {
    "float32": np.float32,
    "float64": np.float64,
}

if COVARIANCE_MATRIX_DTYPE is None:
    COVARIANCE_MATRIX_DTYPE = "float64"

if COVARIANCE_MATRIX_DTYPE in cov_dtype_dict:
    COV_DTYPE = cov_dtype_dict[COVARIANCE_MATRIX_DTYPE]
else:
    COV_DTYPE = np.float64

display(f"Covariance matrix dtype used: {str(COV_DTYPE)}")

"Covariance matrix dtype used: <class 'numpy.float64'>"

# Load data

## Functions

In [10]:
def get_reference_panel_file(directory: Path, file_pattern: str) -> Path:
    files = list(directory.glob(f"*{file_pattern}*.parquet"))
    assert len(files) == 1, f"More than one file was found: {files}"
    return files[0]

In [11]:
# testing
_tmp = get_reference_panel_file(
    conf.PHENOMEXCAN["LD_BLOCKS"]["GTEX_V8_GENOTYPE_DIR"], "chr1.variants"
)
assert _tmp is not None
assert (
    _tmp.name
    == "chr1.variants.parquet"
)

_tmp = get_reference_panel_file(
    conf.PHENOMEXCAN["LD_BLOCKS"]["GTEX_V8_GENOTYPE_DIR"], "_metadata"
)
assert _tmp is not None
assert (
    _tmp.name
    == "variants_metadata.parquet"
)

## SNPs in predictions models

In [12]:
mashr_models_db_files = list(
    conf.PHENOMEXCAN["PREDICTION_MODELS"][EQTL_MODEL].glob("*.db")
)

In [13]:
assert len(mashr_models_db_files) == 49

In [14]:
all_variants_ids = []

for m in mashr_models_db_files:
    print(f"Processing {m.name}")
    tissue = m.name.split(EQTL_MODEL_FILES_PREFIX)[1].split(".db")[0]

    with sqlite3.connect(m) as conn:
        df = pd.read_sql("select gene, varID from weights", conn)
        df["gene"] = df["gene"].apply(lambda x: x.split(".")[0])
        df = df.assign(tissue=tissue)

        all_variants_ids.append(df)

Processing mashr_Thyroid.db
Processing mashr_Artery_Aorta.db


Processing mashr_Heart_Atrial_Appendage.db
Processing mashr_Liver.db
Processing mashr_Heart_Left_Ventricle.db
Processing mashr_Brain_Hippocampus.db
Processing mashr_Testis.db
Processing mashr_Uterus.db
Processing mashr_Adipose_Subcutaneous.db


Processing mashr_Artery_Tibial.db
Processing mashr_Esophagus_Gastroesophageal_Junction.db
Processing mashr_Prostate.db
Processing mashr_Muscle_Skeletal.db
Processing mashr_Brain_Frontal_Cortex_BA9.db
Processing mashr_Whole_Blood.db


Processing mashr_Artery_Coronary.db
Processing mashr_Brain_Cerebellum.db
Processing mashr_Small_Intestine_Terminal_Ileum.db
Processing mashr_Stomach.db
Processing mashr_Brain_Amygdala.db
Processing mashr_Minor_Salivary_Gland.db
Processing mashr_Brain_Putamen_basal_ganglia.db


Processing mashr_Breast_Mammary_Tissue.db
Processing mashr_Adipose_Visceral_Omentum.db
Processing mashr_Brain_Hypothalamus.db
Processing mashr_Brain_Cerebellar_Hemisphere.db
Processing mashr_Brain_Anterior_cingulate_cortex_BA24.db
Processing mashr_Brain_Cortex.db


Processing mashr_Nerve_Tibial.db
Processing mashr_Pancreas.db
Processing mashr_Esophagus_Muscularis.db
Processing mashr_Brain_Substantia_nigra.db
Processing mashr_Skin_Sun_Exposed_Lower_leg.db
Processing mashr_Skin_Not_Sun_Exposed_Suprapubic.db


Processing mashr_Brain_Spinal_cord_cervical_c-1.db
Processing mashr_Spleen.db
Processing mashr_Cells_EBV-transformed_lymphocytes.db
Processing mashr_Vagina.db
Processing mashr_Colon_Sigmoid.db
Processing mashr_Brain_Caudate_basal_ganglia.db
Processing mashr_Brain_Nucleus_accumbens_basal_ganglia.db


Processing mashr_Esophagus_Mucosa.db
Processing mashr_Colon_Transverse.db
Processing mashr_Adrenal_Gland.db
Processing mashr_Lung.db
Processing mashr_Ovary.db
Processing mashr_Cells_Cultured_fibroblasts.db


Processing mashr_Pituitary.db
Processing mashr_Kidney_Cortex.db


In [15]:
all_gene_snps = pd.concat(all_variants_ids, ignore_index=True)

In [16]:
all_gene_snps.shape

(1132714, 3)

In [17]:
all_gene_snps.head()

,gene,varID,tissue
0,ENSG00000180549,chr9_137032508_C_T_b38,Thyroid
1,ENSG00000180549,chr9_137032610_A_G_b38,Thyroid
2,ENSG00000180549,chr9_137032730_G_A_b38,Thyroid
3,ENSG00000107281,chr9_137045741_C_G_b38,Thyroid
4,ENSG00000107281,chr9_137046201_C_A_b38,Thyroid


In [18]:
all_snps_in_models = set(all_gene_snps["varID"].unique())

## MultiPLIER Z

In [19]:
multiplier_z = pd.read_pickle(conf.MULTIPLIER["MODEL_Z_MATRIX_FILE"])

In [20]:
multiplier_z.shape

(6750, 987)

In [21]:
multiplier_z.head()

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,...,LV978,LV979,LV980,LV981,LV982,LV983,LV984,LV985,LV986,LV987
GAS6,0.000000,0.0,0.039438,0.0,0.050476,0.000000,0.0,0.000000,0.590949,0.000000,...,0.050125,0.00000,0.033407,0.000000,0.000000,0.005963,0.347362,0.0,0.000000,0.000000
MMP14,0.000000,0.0,0.000000,0.0,0.070072,0.000000,0.0,0.004904,1.720179,2.423595,...,0.000000,0.00000,0.001007,0.000000,0.035747,0.000000,0.000000,0.0,0.014978,0.000000
DSP,0.000000,0.0,0.000000,0.0,0.000000,0.041697,0.0,0.005718,0.000000,0.000000,...,0.020853,0.00000,0.000000,0.000000,0.000000,0.005774,0.000000,0.0,0.000000,0.416405
MARCKSL1,0.305212,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.161843,0.149471,...,0.027134,0.05272,0.000000,0.030189,0.060884,0.000000,0.000000,0.0,0.000000,0.448480
SPARC,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.014014,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.067779,0.0,0.122417,0.062665


## Reference panel variants metadata

In [22]:
input_file = get_reference_panel_file(REFERENCE_PANEL_DIR, "_metadata")
display(input_file)

PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/data/twas/predixcan/reference_panels/gtex_v8/variants_metadata.parquet')

In [23]:
variants_metadata = pd.read_parquet(input_file, columns=["id"])

In [24]:
variants_metadata.shape

(8880842, 1)

In [25]:
variants_metadata.head()

,id
0,chr1_13550_G_A_b38
1,chr1_14671_G_C_b38
2,chr1_14677_G_A_b38
3,chr1_14933_G_A_b38
4,chr1_16841_G_T_b38


In [26]:
variants_ids_with_genotype = set(variants_metadata["id"])

In [27]:
len(variants_ids_with_genotype)

8880842

In [28]:
list(variants_ids_with_genotype)[:10]

['chr1_246107065_T_C_b38',
 'chr7_73379935_C_T_b38',
 'chr1_56385528_G_A_b38',
 'chr1_21547694_G_A_b38',
 'chr18_68642759_C_T_b38',
 'chr6_161933664_G_A_b38',
 'chr14_95720554_G_T_b38',
 'chr7_134353360_C_T_b38',
 'chr2_237895427_A_G_b38',
 'chr4_138233480_G_A_b38']

In [29]:
del variants_metadata

# How many variants in predictions models are present in the reference panel?

In [30]:
n_snps_in_models = len(all_snps_in_models)
display(n_snps_in_models)

237405

In [31]:
n_snps_in_ref_panel = len(all_snps_in_models.intersection(variants_ids_with_genotype))
display(n_snps_in_ref_panel)

237405

In [32]:
n_snps_in_ref_panel / n_snps_in_models

1.0

# Get final list of genes in MultiPLIER

In [33]:
genes_in_z = [
    Gene(name=gene_name).ensembl_id
    for gene_name in multiplier_z.index
    if gene_name in Gene.GENE_NAME_TO_ID_MAP
]

In [34]:
len(genes_in_z)

6454

In [35]:
genes_in_z[:5]

['ENSG00000183087',
 'ENSG00000157227',
 'ENSG00000096696',
 'ENSG00000175130',
 'ENSG00000113140']

In [36]:
genes_in_z = set(genes_in_z)

In [37]:
len(genes_in_z)

6454

In [38]:
# keep genes in MultiPLIER only
display(all_gene_snps.shape)

all_gene_snps = all_gene_snps[all_gene_snps["gene"].isin(genes_in_z)]

display(all_gene_snps.shape)

(1132714, 3)

(396890, 3)

# (For MultiPLIER genes): How many variants in predictions models are present in the reference panel?

In [39]:
all_snps_in_models_multiplier = set(all_gene_snps["varID"])

n_snps_in_models = len(all_snps_in_models_multiplier)
display(n_snps_in_models)

84708

In [40]:
n_snps_in_ref_panel = len(
    all_snps_in_models_multiplier.intersection(variants_ids_with_genotype)
)
display(n_snps_in_ref_panel)

84708

In [41]:
n_snps_in_ref_panel / n_snps_in_models

1.0

## Preprocess SNPs data

In [42]:
variants_ld_block_df = all_gene_snps[["varID"]].drop_duplicates()

In [43]:
variants_ld_block_df.shape

(84708, 1)

In [44]:
variants_ld_block_df.head()

,varID
6,chr9_137054225_G_A_b38
7,chr9_137054480_G_T_b38
11,chr9_137069838_G_T_b38
12,chr9_137085965_T_C_b38
17,chr9_137185327_T_C_b38


In [45]:
variants_info = variants_ld_block_df["varID"].str.split("_", expand=True)

In [46]:
variants_info.shape

(84708, 5)

In [47]:
assert variants_ld_block_df.shape[0] == variants_info.shape[0]

In [48]:
variants_ld_block_df = variants_ld_block_df.join(variants_info)[["varID", 0, 1, 2, 3]]

In [49]:
assert variants_ld_block_df.shape[0] == variants_info.shape[0]

In [50]:
variants_ld_block_df.head()

,varID,0,1,2,3
6,chr9_137054225_G_A_b38,chr9,137054225,G,A
7,chr9_137054480_G_T_b38,chr9,137054480,G,T
11,chr9_137069838_G_T_b38,chr9,137069838,G,T
12,chr9_137085965_T_C_b38,chr9,137085965,T,C
17,chr9_137185327_T_C_b38,chr9,137185327,T,C


In [51]:
variants_ld_block_df = variants_ld_block_df.rename(
    columns={
        0: "chr",
        1: "position",
        2: "ref_allele",
        3: "eff_allele",
    }
)

In [52]:
variants_ld_block_df["chr"] = variants_ld_block_df["chr"].apply(lambda x: int(x[3:]))

In [53]:
variants_ld_block_df["position"] = variants_ld_block_df["position"].astype(int)

In [54]:
variants_ld_block_df.shape

(84708, 5)

In [55]:
variants_ld_block_df.head()

,varID,chr,position,ref_allele,eff_allele
6,chr9_137054225_G_A_b38,9,137054225,G,A
7,chr9_137054480_G_T_b38,9,137054480,G,T
11,chr9_137069838_G_T_b38,9,137069838,G,T
12,chr9_137085965_T_C_b38,9,137085965,T,C
17,chr9_137185327_T_C_b38,9,137185327,T,C


In [56]:
variants_ld_block_df.dtypes

varID         object
chr            int64
position       int64
ref_allele    object
eff_allele    object
dtype: object

# Covariance for each chromosome block

## Functions

In [57]:
def covariance(df, dtype):
    n = df.shape[0]
    df = df.sub(df.mean(), axis=1).astype(dtype)
    return df.T.dot(df) / (n - 1)

In [58]:
# testing
rs = np.random.RandomState(0)

_test_data = pd.DataFrame(rs.normal(size=(50, 5)), columns=[f"c{i}" for i in range(5)])

# float64
pd.testing.assert_frame_equal(
    covariance(_test_data, np.float64),
    _test_data.cov(),
    rtol=1e-10,
    atol=1e-10,
    check_dtype=True,
)

# float32
pd.testing.assert_frame_equal(
    covariance(_test_data, np.float32),
    _test_data.cov(),
    rtol=1e-5,
    atol=1e-8,
    check_dtype=False,
)

del _test_data

In [59]:
def compute_snps_cov(snps_df):
    assert snps_df["chr"].unique().shape[0] == 1
    chromosome = snps_df["chr"].unique()[0]

    # keep variants only present in genotype
    snps_ids = list(set(snps_df["varID"]).intersection(variants_ids_with_genotype))

    chromosome_file = get_reference_panel_file(
        REFERENCE_PANEL_DIR, f"chr{chromosome}.variants"
    )
    snps_genotypes = pd.read_parquet(chromosome_file, columns=snps_ids)

    return covariance(snps_genotypes, COV_DTYPE)

In [60]:
# testing
_tmp_snps = variants_ld_block_df[variants_ld_block_df["chr"] == 22]
assert _tmp_snps.shape[0] > 0

In [61]:
_tmp_snps.shape

(2687, 5)

In [62]:
n_expected = len(set(_tmp_snps["varID"]).intersection(variants_ids_with_genotype))
display(n_expected)

2687

In [63]:
_tmp = compute_snps_cov(_tmp_snps)

In [64]:
assert _tmp.shape == (n_expected, n_expected)
assert not _tmp.isna().any().any()

In [65]:
del _tmp_snps, _tmp

## Compute covariance and save

In [66]:
output_file_name_template = "snps_chr_blocks_cov.h5"

output_file = OUTPUT_DIR_BASE / output_file_name_template
display(output_file)

PosixPath('/home/miltondp/projects/phenoplier/clean_orig_base/phenoplier-gwas-40pcs/results/gls/gene_corrs/reference_panels/gtex_v8/mashr/snps_chr_blocks_cov.h5')

In [67]:
with pd.HDFStore(output_file, mode="w", complevel=4) as store:
    pbar = tqdm(
        variants_ld_block_df.groupby("chr"),
        ncols=100,
        total=variants_ld_block_df["chr"].unique().shape[0],
    )

    store["metadata"] = variants_ld_block_df

    for grp_name, grp_data in pbar:
        pbar.set_description(f"{grp_name} {grp_data.shape}")
        snps_cov = compute_snps_cov(grp_data)  # .astype(COV_DTYPE)
        assert not snps_cov.isna().any().any()
        store[f"chr{grp_name}"] = snps_cov

        del snps_cov
        store.flush()

        gc.collect()

  0%|                                                                        | 0/22 [00:00<?, ?it/s]

1 (8130, 5):   0%|                                                           | 0/22 [00:00<?, ?it/s]

1 (8130, 5):   5%|██▏                                             | 1/22 [02:57<1:02:01, 177.20s/it]

2 (5857, 5):   5%|██▏                                             | 1/22 [02:57<1:02:01, 177.20s/it]

2 (5857, 5):   9%|████▌                                             | 2/22 [05:25<53:18, 159.94s/it]

...

21 (1242, 5):  91%|████████████████████████████████████████████▌    | 20/22 [28:21<01:45, 52.54s/it]

21 (1242, 5):  95%|██████████████████████████████████████████████▊  | 21/22 [28:43<00:43, 43.49s/it]

22 (2687, 5):  95%|██████████████████████████████████████████████▊  | 21/22 [28:43<00:43, 43.49s/it]

22 (2687, 5): 100%|█████████████████████████████████████████████████| 22/22 [29:13<00:00, 39.40s/it]

22 (2687, 5): 100%|█████████████████████████████████████████████████| 22/22 [29:13<00:00, 79.70s/it]

# Testing

In [68]:
_tmp = variants_ld_block_df[variants_ld_block_df["chr"] == 1]

In [69]:
_tmp.shape

(8130, 5)

In [70]:
assert _tmp.shape[0] > 0

In [71]:
n_expected = len(set(_tmp["varID"]).intersection(variants_ids_with_genotype))
display(n_expected)
assert n_expected > 0

8130

In [72]:
with pd.HDFStore(output_file, mode="r") as store:
    df = store["chr1"]
    assert df.shape == (n_expected, n_expected)
    assert not df.isna().any().any()